Goal of Project: How does GPT-2 represent multi-token compound concepts internally?

Setup: LLMs have a fixed residual stream dimension (~768 for GPT-2), but there are far more concepts in the world than dimensions. So when the model sees "washing machine", does it have a dedicated direction for that compound concept, or does it just store "washing" and rely on context/statistics to predict "machine"?

PMI is used as a proxy for "how compound-like" a bigram is (high PMI means the two words appear together more than chance, suggesting they function as a unit).         
  
Experiments check: do high-PMI pairs get treated differently inside the model than low-PMI pairs? 
1. Do their residual stream representations look more like a blend of components (additive) or something unique?
2. Do they activate distinct SAE features vs. just the union of each word's features?
3. Does comp1 strongly prime comp2, and does that scale with PMI?                          

Extension: 
    
Data
- Wikipedia top-1000 bigrams (filtered) by PMI

Experiments 1/2/3a
- Replaces compositionality labels with PMI as the continuous variable
- Additional plots 

Experiment 3b 
- Groups bigrams by their modifier (comp1) (keeping only modifiers that appear with 2+ different compounds)    
- Within each modifier group, checks whether higher PMI consistently predicts better rank and higher probability
- Splits results into "good groups" (PMI and rank agree) vs "bad groups" (counterexamples where higher PMI doesn't give better rank)

In [ ]:
import transformers
import huggingface_hub
import re
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from collections import Counter, defaultdict
from nltk.corpus import stopwords
from nltk import pos_tag
from transformer_lens import HookedTransformer
from manual_sae import ManualSAE

## Load + Filter/Clean Dataset

Loading & Cleaning:
- Load Wikipedia (20220301.en) from HuggingFace

Tokenization & Filtering:
- tokenize_with_punctuation(): split text into tokens wiht punctuation
- is_two_token(): keep only phrases that are exactly 2 GPT-2 tokens (so "washing machine" counts as one unit)
- STOPWORDS (filter junk phrases) and POS-tag tokens with NLTK (filter bigrams by POS patterns (JJ-NN, NN-NN, VBG-NN, etc.), so only adjective-noun, verb-noun, and noun-noun type pairs)

Counting:
- get_counts(): count unigrams and bigrams across up to 5000 Wikipedia docs
- compute_pmi_scores(): compute PMI for each bigram using unigram/bigram counts

Output:
- bigrams(): full pipeline returning bigrams sorted by PMI
- Take top 1000 by PMI for experiments

In [ ]:
dataset = load_dataset("wikipedia", "20220301.en", split="train")

In [ ]:
DEVICE = "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
tokenizer = model.tokenizer

def wiki_texts(dataset, max_docs=5000):
    for i, item in enumerate(dataset):
        if i >= max_docs:
            break
        yield item["text"]

def clean_text(text):
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_with_punctuation(text):
    """
    Ex: "Hello, World." -> ["Hello", ",", "World", "."]
    """
    return re.findall(r"[A-Za-z]+|[^\w\s]", text)

def is_two_token(w1, w2, tokenizer):
    t1 = tokenizer.encode(" " + w1, add_special_tokens=False)
    t2 = tokenizer.encode(" " + w2, add_special_tokens=False)
    phrase = tokenizer.encode(" " + w1 + " " + w2, add_special_tokens=False)

    return len(t1) == 1 and len(t2) == 1 and len(phrase) == 2

# filter for good bigrams 
STOPWORDS = set(stopwords.words("english"))

ALLOWED_PATTERNS = {
    ("JJ", "NN"), ("JJ", "NNS"),
    ("NN", "NN"), ("NN", "NNS"),
    ("NNS", "NN"), ("NNS", "NNS"),
    ("VBG", "NN"), ("VBG", "NNS")
}

def get_counts(dataset, max_docs=5000):
    unigram_counts = Counter()
    bigram_counts = Counter()
    total_words = 0

    corpus = wiki_texts(dataset, max_docs)

    for text in corpus:
        text = clean_text(text)
        tokens = tokenize_with_punctuation(text)

        if not tokens:
            continue

        words = [tok for tok in tokens if tok.isalpha()]
        if not words:
            continue

        tagged_words = pos_tag(words)

        lower_words = [w.lower() for w in words]
        unigram_counts.update(lower_words)
        total_words += len(lower_words)

        word_token_positions = []
        for idx, tok in enumerate(tokens):
            if tok.isalpha():
                word_token_positions.append(idx)

        token_pos_to_word_idx = {tok_pos: word_idx for word_idx, tok_pos in enumerate(word_token_positions)}

        for i in range(len(tokens) - 1):
            # only count bigrams where two adjacent tokens are both words
            if not (tokens[i].isalpha() and tokens[i + 1].isalpha()):
                continue

            w1 = tokens[i]
            w2 = tokens[i + 1]

            word_idx_1 = token_pos_to_word_idx[i]
            word_idx_2 = token_pos_to_word_idx[i + 1]

            t1 = tagged_words[word_idx_1][1]
            t2 = tagged_words[word_idx_2][1]

            w1_lower = w1.lower()
            w2_lower = w2.lower()

            # remove junk phrases / proper nouns
            if w1_lower in STOPWORDS or w2_lower in STOPWORDS:
                continue
            if len(w1_lower) < 3 or len(w2_lower) < 3:
                continue
            if t1 in {"NNP", "NNPS"} or t2 in {"NNP", "NNPS"}:
                continue
            if (t1, t2) not in ALLOWED_PATTERNS:
                continue

            # check if valid GPT-2 2-token phrase
            if not is_two_token(w1_lower, w2_lower, tokenizer):
                continue

            bigram_counts[(w1_lower, w2_lower)] += 1

    return unigram_counts, bigram_counts, total_words

def compute_pmi_scores(unigram_counts, bigram_counts, total_words,
                       min_bigram_count=25, min_unigram_count=500):
    results = []

    for (w1, w2), c12 in bigram_counts.items():
        c1 = unigram_counts[w1]
        c2 = unigram_counts[w2]

        if c12 < min_bigram_count:
            continue
        if c1 < min_unigram_count or c2 < min_unigram_count:
            continue
        
        frequency = c12 / total_words
        pmi = math.log((c12 * total_words) / (c1 * c2))

        results.append({
            "w1": w1,
            "w2": w2,
            "bigram_count": c12,
            "count_w1": c1,
            "count_w2": c2,
            "pmi": pmi, 
            "frequency": frequency
        })

    results.sort(key=lambda x: x["pmi"], reverse=True)
    return results

def bigrams(dataset, max_docs=5000, min_bigram_count=25, min_unigram_count=500):
    unigram_counts, bigram_counts, total_words = get_counts(dataset, max_docs=max_docs)

    results = compute_pmi_scores(
        unigram_counts=unigram_counts,
        bigram_counts=bigram_counts,
        total_words=total_words,
        min_bigram_count=min_bigram_count,
        min_unigram_count=min_unigram_count
    )

    return results

In [ ]:
bigrams = bigrams(dataset, max_docs=5000)

In [ ]:
bigrams

In [ ]:
n = 1000
top_n_bigrams = bigrams[:n]

In [ ]:
top_n_bigrams

## Experiment 1

Run the compound ("washing machine"), comp1 ("washing"), and comp2 ("machine") through GPT-2 separately and grab the residual stream at each layer.

Compute cosine similarity of comp2 (" machine") between the compound's representation and: (1) word2 alone, (2) the additive average of word1+word2, (3) word1 alone. 

Checks: Does a compound's internal representation look more like its components as PMI increases? Is it just the average of its parts, or something distinct? 

In [ ]:
bigram_data = [(bigram["pmi"], bigram["w1"], bigram["w2"], f'{bigram["w1"]} {bigram["w2"]}') for bigram in top_n_bigrams]

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 1: Residual Stream Cosine Similarity Analysis")
print("="*60)

# Get residual stream after each layer layers for specified token positions
def get_residual_activations(text):
    tokens = tokenizer.encode(text)
    input_ids = torch.tensor([tokens]).to(DEVICE)

    with torch.no_grad():
        _, cache = model.run_with_cache(input_ids)

    activations = {}
    for layer in range(model.cfg.n_layers):
        # Get residual stream
        key = f"blocks.{layer}.hook_resid_post"
        act = cache[key][0]  # [seq_len, d_model]
        activations[layer] = act.cpu()

    return activations, tokens

def cosine_sim(a, b):
    return torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# Save results 
pmi_vals = []
compounds = []

comp2_cos_sims = []
additive_cos_sims = []
comp1_cos_sims = []

for bigram in bigram_data:
    pmi, comp1, comp2, compound = bigram
    
    # compound ("the washing machine")
    compound_text = f"The {compound}"
    compound_acts, compound_toks = get_residual_activations(compound_text)
    
    # comp1 ("the washing")
    comp1_text = f"The {comp1}"
    comp1_acts, comp1_toks = get_residual_activations(comp1_text)

    # comp2 ("the machine")
    comp2_text = f"The {comp2}"
    comp2_acts, comp2_toks = get_residual_activations(comp2_text)
    
    # Compute layer-wise cosine similarities
    layer_cos_sims = {"compound_vs_comp2": [], "compound_vs_additive": [],
                  "compound_vs_comp1": []}

    for layer in range(model.cfg.n_layers):
        compound_repr = compound_acts[layer][-1]
        comp1_repr = comp1_acts[layer][-1]
        comp2_repr = comp2_acts[layer][-1]
        additive_repr = (comp1_repr + comp2_repr) / 2.0

        layer_cos_sims["compound_vs_comp2"].append(cosine_sim(compound_repr, comp2_repr))
        layer_cos_sims["compound_vs_additive"].append(cosine_sim(compound_repr, additive_repr))
        layer_cos_sims["compound_vs_comp1"].append(cosine_sim(compound_repr, comp1_repr))
    
    # Update 
    pmi_vals.append(pmi)
    compounds.append(compound)
    
    comp2_cos_sims.append(layer_cos_sims['compound_vs_comp2'])
    additive_cos_sims.append(layer_cos_sims['compound_vs_additive'])
    comp1_cos_sims.append(layer_cos_sims['compound_vs_comp1'])
    
    print(f"{compound} (layer 11): comp2_cos_sim={layer_cos_sims['compound_vs_comp2'][-1]:.3f}, "
          f"add_cos_sim={layer_cos_sims['compound_vs_additive'][-1]:.3f}, "
          f"comp1_cos_sim={layer_cos_sims['compound_vs_comp1'][-1]:.3f}")

In [ ]:
# Plot 

plt.style.use("seaborn-v0_8-whitegrid")

x = np.array(pmi_vals)
comp2_cos_sims = np.array(comp2_cos_sims)
additive_cos_sims = np.array(additive_cos_sims)
comp1_cos_sims = np.array(comp1_cos_sims)

COLORS = {
    "comp2": "#1f77b4",   # deep blue
    "add": "#6c757d",     # medium gray
    "comp1": "#adb5bd"    # light gray
}

for layer in range(model.cfg.n_layers):
    y_comp2 = comp2_cos_sims[:, layer]
    y_add = additive_cos_sims[:, layer]
    y_comp1 = comp1_cos_sims[:, layer]

    plt.figure(figsize=(9, 6), dpi=400)

    plt.scatter(x, y_comp2, alpha=0.7, s=25, color=COLORS["comp2"], label="vs comp2")
    plt.scatter(x, y_add, alpha=0.7, s=25, color=COLORS["add"], label="vs additive")
    plt.scatter(x, y_comp1, alpha=0.7, s=25, color=COLORS["comp1"], label="vs comp1")

    m2, b2 = np.polyfit(x, y_comp2, 1)
    ma, ba = np.polyfit(x, y_add, 1)
    m1, b1 = np.polyfit(x, y_comp1, 1)

    x_line = np.linspace(x.min(), x.max(), 200)

    plt.plot(x_line, m2 * x_line + b2, color=COLORS["comp2"], linewidth=2)
    plt.plot(x_line, ma * x_line + ba, color=COLORS["add"], linewidth=2)
    plt.plot(x_line, m1 * x_line + b1, color=COLORS["comp1"], linewidth=2)

    plt.xlabel("PMI", fontsize=12)
    plt.ylabel("Cosine Similarity", fontsize=12)
    plt.title(f"Layer {layer}", fontsize=14)

    plt.legend(frameon=False, fontsize=10)

    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
# Layer 11

print(f"cos sim compound <-> comp2: {np.mean([cos_sims[-1] for cos_sims in comp2_cos_sims])}")
print(f"cos sim compound <-> additive: {np.mean([cos_sims[-1] for cos_sims in additive_cos_sims])}")
print(f"cos sim compound <-> comp1: {np.mean([cos_sims[-1] for cos_sims in comp1_cos_sims])}")

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

x = np.array(pmi_vals)

num_layers = comp2_cos_sims.shape[1]

for layer in range(num_layers):
    y_comp2 = comp2_cos_sims[:, layer]
    y_add = additive_cos_sims[:, layer]
    y_comp1 = comp1_cos_sims[:, layer]

    num_bins = 5
    bins = np.linspace(x.min(), x.max(), num_bins + 1)
    bin_indices = np.digitize(x, bins) - 1
    bin_indices[bin_indices == num_bins] = num_bins - 1

    comp2_means, add_means, comp1_means = [], [], []
    comp2_stds, add_stds, comp1_stds = [], [], []
    labels = []

    for b in range(num_bins):
        mask = bin_indices == b

        if np.sum(mask) == 0:
            continue

        comp2_means.append(y_comp2[mask].mean())
        add_means.append(y_add[mask].mean())
        comp1_means.append(y_comp1[mask].mean())

        comp2_stds.append(y_comp2[mask].std())
        add_stds.append(y_add[mask].std())
        comp1_stds.append(y_comp1[mask].std())

        labels.append(f"{bins[b]:.1f}-{bins[b+1]:.1f}")

    x_pos = np.arange(len(labels))
    width = 0.25

    plt.figure(figsize=(9, 6), dpi=400)

    plt.bar(
        x_pos - width, comp2_means, width,
        yerr=comp2_stds, capsize=4,
        color=COLORS["comp2"], label="vs comp2"
    )

    plt.bar(
        x_pos, add_means, width,
        yerr=add_stds, capsize=4,
        color=COLORS["add"], label="vs additive"
    )

    plt.bar(
        x_pos + width, comp1_means, width,
        yerr=comp1_stds, capsize=4,
        color=COLORS["comp1"], label="vs comp1"
    )

    plt.xticks(x_pos, labels)
    plt.xlabel("PMI range")
    plt.ylabel(f"Average Cosine similarity (Layer {layer})")
    plt.title(f"Layer {layer}")

    plt.legend(frameon=True)
    plt.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

## Experiment 2

Load sparse autoencoders for layers 1, 6, 11. For each bigram, you get the active SAE features for the compound vs. each component, then compute Jaccard similarity, feature overlap ratios, and cosine similarity of the full SAE activation vectors.

Checks: Do the compound and its components share the same sparse features in the model's dictionary? Does feature overlap correlate with PMI?         

In [ ]:
bigram_data = [(bigram["pmi"], bigram["w1"], bigram["w2"], f'{bigram["w1"]} {bigram["w2"]}') for bigram in top_n_bigrams]

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 2: SAE Feature Analysis")
print("="*60)

# Load SAEs
TARGET_LAYERS = [1, 6, 11]
saes = {}
for layer in TARGET_LAYERS:
    try:
        sae, cfg = ManualSAE.from_pretrained(layer=layer, device=DEVICE)
        saes[layer] = sae
    except Exception as e:
        print(f"Failed to load SAE for layer {layer}: {e}")

def get_sae_features(text, sae, layer, token_position=-1):
    tokens = tokenizer.encode(text)
    input_ids = torch.tensor([tokens]).to(DEVICE)

    with torch.no_grad():
        _, cache = model.run_with_cache(input_ids)

    key = f"blocks.{layer}.hook_resid_pre"
    activation = cache[key][0, token_position]   # [d_model]

    with torch.no_grad():
        feature_acts = sae.encode(activation.unsqueeze(0))  # [1, d_sae]

    feature_acts = feature_acts.squeeze(0).cpu()
    active_features = torch.nonzero(feature_acts > 0).squeeze(-1).tolist()
    active_values = feature_acts[feature_acts > 0].tolist()

    if isinstance(active_features, int):
        active_features = [active_features]
        active_values = [active_values]

    return active_features, active_values, feature_acts

def jaccard_similarity(set1, set2):
    s1, s2 = set(set1), set(set2)
    if len(s1) == 0 and len(s2) == 0:
        return 1.0
    intersection = len(s1 & s2)
    union = len(s1 | s2)
    return intersection / union if union > 0 else 0.0

exp2_results = {}

for bigram in bigram_data:
    pmi, comp1, comp2, compound = bigram

    compound_text = f"The {compound}"
    comp1_text = f"The {comp1}"
    comp2_text = f"The {comp2}"

    exp2_results[compound] = {
        "pmi": pmi,
        "comp1": comp1,
        "comp2": comp2,
        "layers": {}
    }

    for layer in TARGET_LAYERS:
        if layer not in saes:
            print(f"Skipping layer {layer} for {compound} (SAE not loaded)")
            continue

        sae = saes[layer]

        # compound
        compound_feats, compound_vals, compound_acts = get_sae_features(compound_text, sae, layer, token_position=-1)
        # comp2
        comp2_feats, comp2_vals, comp2_acts = get_sae_features(comp2_text, sae, layer, token_position=-1)
        # comp1
        comp1_feats, comp1_vals, comp1_acts = get_sae_features(comp1_text, sae, layer, token_position=-1)

        # sets
        compound_set = set(compound_feats)
        comp2_set = set(comp2_feats)
        comp1_set = set(comp1_feats)

        # Jaccard similarities
        jacc_compound_comp2 = jaccard_similarity(compound_feats, comp2_feats)
        jacc_compound_comp1 = jaccard_similarity(compound_feats, comp1_feats)
        jacc_compound_union = jaccard_similarity(compound_feats, list(comp2_set | comp1_set))

        # Feature overlap ratios
        overlap_with_comp2 = len(compound_set & comp2_set) / len(compound_set) if compound_set else 0.0
        overlap_with_comp1 = len(compound_set & comp1_set) / len(compound_set) if compound_set else 0.0
        unique_to_compound = len(compound_set - comp2_set - comp1_set) / len(compound_set) if compound_set else 0.0

        # Cosine similarity of SAE activation vectors
        cos_sim_comp2 = cosine_sim(compound_acts, comp2_acts)
        cos_sim_comp1 = cosine_sim(compound_acts, comp1_acts)
        cos_sim_union = cosine_sim(compound_acts, (comp1_acts + comp2_acts) / 2.0)

        # store everything
        exp2_results[compound]["layers"][layer] = {
            # PMI
            "pmi": pmi, 
            
            # raw feature info
            "compound_feats": compound_feats,
            "compound_vals": compound_vals,
            "comp1_feats": comp1_feats,
            "comp1_vals": comp1_vals,
            "comp2_feats": comp2_feats,
            "comp2_vals": comp2_vals,

            # Jaccard similarities
            "jacc_compound_comp2": jacc_compound_comp2,
            "jacc_compound_comp1": jacc_compound_comp1,
            "jacc_compound_union": jacc_compound_union,

            # Feature overlap ratios
            "overlap_with_comp2": overlap_with_comp2,
            "overlap_with_comp1": overlap_with_comp1,
            "unique_to_compound": unique_to_compound,

            # Cosine similarities
            "cos_sim_comp2": cos_sim_comp2,
            "cos_sim_comp1": cos_sim_comp1,
            "cos_sim_union": cos_sim_union,

            # counts
            "n_compound_feats": len(compound_feats),
            "n_comp1_feats": len(comp1_feats),
            "n_comp2_feats": len(comp2_feats),
        }

        print(
            f"{compound} (layer {layer}): "
            f"jacc_compound_comp2={jacc_compound_comp2:.3f}, "
            f"jacc_compound_comp1={jacc_compound_comp1:.3f}, "
            f"jacc_compound_union={jacc_compound_union:.3f}, "
            f"overlap_with_comp2={overlap_with_comp2:.3f}, "
            f"overlap_with_comp1={overlap_with_comp1:.3f}, "
            f"unique_to_compound={unique_to_compound:.3f}, "
            f"cos_sim_comp2={cos_sim_comp2:.3f}, "
            f"cos_sim_comp1={cos_sim_comp1:.3f}, "
            f"cos_sim_union={cos_sim_union:.3f}"
        )

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

COLORS = {
    "primary": "#6f42c1",   # purple
    "dark": "#495057",      # dark gray
    "light": "#ced4da"      # light gray
}

TARGET_LAYERS = [1, 6, 11]

avg_jacc_head = []
avg_jacc_mod = []
avg_jacc_union = []

avg_overlap_head = []
avg_overlap_mod = []
avg_unique = []

avg_cos_head = []
avg_cos_mod = []
avg_cos_add = []

for layer in TARGET_LAYERS:
    compounds = [c for c in exp2_results if layer in exp2_results[c]["layers"]]

    avg_jacc_head.append(np.mean([exp2_results[c]["layers"][layer]["jacc_compound_comp2"] for c in compounds]))
    avg_jacc_mod.append(np.mean([exp2_results[c]["layers"][layer]["jacc_compound_comp1"] for c in compounds]))
    avg_jacc_union.append(np.mean([exp2_results[c]["layers"][layer]["jacc_compound_union"] for c in compounds]))

    avg_overlap_head.append(np.mean([exp2_results[c]["layers"][layer]["overlap_with_comp2"] for c in compounds]))
    avg_overlap_mod.append(np.mean([exp2_results[c]["layers"][layer]["overlap_with_comp1"] for c in compounds]))
    avg_unique.append(np.mean([exp2_results[c]["layers"][layer]["unique_to_compound"] for c in compounds]))

    avg_cos_head.append(np.mean([exp2_results[c]["layers"][layer]["cos_sim_comp2"] for c in compounds]))
    avg_cos_mod.append(np.mean([exp2_results[c]["layers"][layer]["cos_sim_comp1"] for c in compounds]))
    avg_cos_add.append(np.mean([exp2_results[c]["layers"][layer]["cos_sim_union"] for c in compounds]))

x = np.arange(len(TARGET_LAYERS))
width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Jaccard
axes[0].bar(x - width, avg_jacc_union, width, color=COLORS["primary"], label="vs union")
axes[0].bar(x,         avg_jacc_head,  width, color=COLORS["dark"],   label="vs head")
axes[0].bar(x + width, avg_jacc_mod,   width, color=COLORS["light"],  label="vs modifier")
axes[0].set_xticks(x)
axes[0].set_xticklabels(TARGET_LAYERS)
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Average Jaccard similarity")
axes[0].set_title("Average Jaccard by Layer")
axes[0].set_ylim(0, 1)
axes[0].legend()

# Feature overlap
axes[1].bar(x - width, avg_unique,       width, color=COLORS["primary"], label="Unique to compound")
axes[1].bar(x,         avg_overlap_head, width, color=COLORS["dark"],   label="Overlap with head")
axes[1].bar(x + width, avg_overlap_mod,  width, color=COLORS["light"],  label="Overlap with modifier")
axes[1].set_xticks(x)
axes[1].set_xticklabels(TARGET_LAYERS)
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Average feature fraction")
axes[1].set_title("Average Feature Overlap by Layer")
axes[1].set_ylim(0, 1)
axes[1].legend()

# Cosine sim 
axes[2].bar(x - width, avg_cos_add, width, color=COLORS["primary"], label="vs additive")
axes[2].bar(x,         avg_cos_head,  width, color=COLORS["dark"],   label="vs head")
axes[2].bar(x + width, avg_cos_mod,   width, color=COLORS["light"],  label="vs modifier")
axes[2].set_xticks(x)
axes[2].set_xticklabels(TARGET_LAYERS)
axes[2].set_xlabel("Layer")
axes[2].set_ylabel("Average cosine similarity")
axes[2].set_title("Average SAE Cosine by Layer")
axes[2].set_ylim(0, 1)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Layer 11

print(f"average SAE jacc sim compound <-> comp1 union comp2: {avg_jacc_union[-1]}")
print(f"average SAE jacc sim compound <-> comp2: {avg_jacc_head[-1]}")
print(f"average SAE jacc sim compound <-> comp1: {avg_jacc_mod[-1]}\n")

print(f"average SAE unique to compound: {avg_unique[-1]}")
print(f"average SAE overlap compound <-> comp2: {avg_overlap_head[-1]}")
print(f"average SAE overlap compound <-> comp1: {avg_overlap_mod[-1]}\n")

print(f"average SAE cos sim compound <-> additive: {avg_cos_add[-1]}")
print(f"average SAE cos sim compound <-> comp2: {avg_cos_head[-1]}")
print(f"average SAE cos sim compound <-> comp1: {avg_cos_mod[-1]}")

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

COLORS = {
    "primary": "#6f42c1",   # purple
    "dark": "#495057",      # dark gray
    "light": "#ced4da"      # light gray
}

TARGET_LAYERS = [1, 6, 11]

for layer in TARGET_LAYERS:
    compounds = [c for c in exp2_results if layer in exp2_results[c]["layers"]]

    x = np.array([exp2_results[c]["pmi"] for c in compounds])

    # Jaccard
    y_jacc_head = np.array([exp2_results[c]["layers"][layer]["jacc_compound_comp2"] for c in compounds])
    y_jacc_mod = np.array([exp2_results[c]["layers"][layer]["jacc_compound_comp1"] for c in compounds])
    y_jacc_union = np.array([exp2_results[c]["layers"][layer]["jacc_compound_union"] for c in compounds])

    # Feature overlap
    y_overlap_head = np.array([exp2_results[c]["layers"][layer]["overlap_with_comp2"] for c in compounds])
    y_overlap_mod = np.array([exp2_results[c]["layers"][layer]["overlap_with_comp1"] for c in compounds])
    y_unique = np.array([exp2_results[c]["layers"][layer]["unique_to_compound"] for c in compounds])

    # Cosine sim 
    y_cos_head = np.array([exp2_results[c]["layers"][layer]["cos_sim_comp2"] for c in compounds])
    y_cos_mod = np.array([exp2_results[c]["layers"][layer]["cos_sim_comp1"] for c in compounds])
    y_cos_union = np.array([exp2_results[c]["layers"][layer]["cos_sim_union"] for c in compounds])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Jaccard vs PMI
    axes[0].scatter(x, y_jacc_union, color=COLORS["primary"], label="vs union", alpha=0.8, s=60)
    axes[0].scatter(x, y_jacc_head, color=COLORS["dark"], label="vs head", alpha=0.8, s=60)
    axes[0].scatter(x, y_jacc_mod, color=COLORS["light"], label="vs modifier", alpha=0.9, s=60)

    for y, color in [
        (y_jacc_union, COLORS["primary"]),
        (y_jacc_head, COLORS["dark"]),
        (y_jacc_mod, COLORS["light"])
    ]:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        axes[0].plot(x_line, m * x_line + b, linestyle="--", color=color)

    axes[0].set_title(f"Layer {layer}: Jaccard vs PMI")
    axes[0].set_xlabel("PMI")
    axes[0].set_ylabel("Jaccard similarity")
    axes[0].set_ylim(0, 1)
    axes[0].legend()

    # Feature overlap vs PMI
    axes[1].scatter(x, y_unique, color=COLORS["primary"], label="Unique to compound", alpha=0.8, s=60)
    axes[1].scatter(x, y_overlap_head, color=COLORS["dark"], label="Overlap with head", alpha=0.8, s=60)
    axes[1].scatter(x, y_overlap_mod, color=COLORS["light"], label="Overlap with modifier", alpha=0.9, s=60)

    for y, color in [
        (y_unique, COLORS["primary"]),
        (y_overlap_head, COLORS["dark"]),
        (y_overlap_mod, COLORS["light"])
    ]:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        axes[1].plot(x_line, m * x_line + b, linestyle="--", color=color)

    axes[1].set_title(f"Layer {layer}: Overlap / Unique vs PMI")
    axes[1].set_xlabel("PMI")
    axes[1].set_ylabel("Feature fraction")
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    # Cosine sim vs PMI
    axes[2].scatter(x, y_cos_union, color=COLORS["primary"], label="vs additive", alpha=0.8, s=60)
    axes[2].scatter(x, y_cos_head, color=COLORS["dark"], label="vs head", alpha=0.8, s=60)
    axes[2].scatter(x, y_cos_mod, color=COLORS["light"], label="vs modifier", alpha=0.9, s=60)

    for y, color in [
        (y_cos_union, COLORS["primary"]),
        (y_cos_head, COLORS["dark"]),
        (y_cos_mod, COLORS["light"])
    ]:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        axes[2].plot(x_line, m * x_line + b, linestyle="--", color=color)

    axes[2].set_title(f"Layer {layer}: Cosine vs PMI")
    axes[2].set_xlabel("PMI")
    axes[2].set_ylabel("Cosine similarity")
    axes[2].set_ylim(0, 1)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## Experiment 3a

For each bigram, compare P(comp2 | "The comp1") vs. the baseline P(comp2 | "The"). Compute a priming ratio and log-priming ratio, and scatter plot them against PMI.

Checks: Does seeing comp1 make the model more likely to predict comp2? Does this scale with PMI?

In [ ]:
bigram_data = [(bigram["pmi"], bigram["w1"], bigram["w2"], f'{bigram["w1"]} {bigram["w2"]}') for bigram in top_n_bigrams]

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 3a: Prediction Analysis")
print("="*60)

exp3a_results = {}

pmi_vals = []
prob_vals = []
pr_vals = []
labels = []
ranks_modifier = []

for bigram in bigram_data:
    pmi, comp1, comp2, compound = bigram
    
    prompt_with_modifier = f"The {comp1}"
    prompt_baseline = "The"
    
    with torch.no_grad():
        # Get logits for modifier
        tokens_modifier = tokenizer.encode(prompt_with_modifier)
        logits_modifier = model(torch.tensor([tokens_modifier]).to(DEVICE))
        probs_modifier = torch.softmax(logits_modifier[0, -1], dim=-1)
        top_probs_modifier, top_indices_modifier = torch.topk(probs_modifier, k=5)
        top_tokens_modifier = [tokenizer.decode([idx]) for idx in top_indices_modifier]
#         print(top_probs_modifier, top_tokens_modifier)

        # Get logits for baseline
        tokens_baseline = tokenizer.encode(prompt_baseline)
        logits_baseline = model(torch.tensor([tokens_baseline]).to(DEVICE))
        probs_baseline = torch.softmax(logits_baseline[0, -1], dim=-1)
        top_probs_baseline, top_indices_baseline = torch.topk(probs_baseline, k=5)
        top_tokens_baseline = [tokenizer.decode([idx]) for idx in top_indices_baseline]
#         print(top_probs_modifier, top_tokens_baseline)
    
    # Get probability of head noun token
    head_token_id = tokenizer.encode(" " + comp2)[0]  # First token of head noun with space

    p_head_given_modifier = probs_modifier[head_token_id].item()
    p_head_baseline = probs_baseline[head_token_id].item()
    
    sorted_probs_modifier, sorted_indices_modifier = torch.sort(probs_modifier, descending=True)
    rank_modifier = (sorted_indices_modifier == head_token_id).nonzero().item() + 1
    
    sorted_probs_baseline, sorted_indices_baseline = torch.sort(probs_baseline, descending=True)
    rank_baseline = (sorted_indices_baseline == head_token_id).nonzero().item() + 1
    
    # Priming ratio
    priming_ratio = p_head_given_modifier / p_head_baseline if p_head_baseline > 0 else float('inf')
    
    exp3a_results[(comp1, comp2)] = {
        "pmi": pmi,
        "p_head_given_modifier": p_head_given_modifier,
        "p_head_baseline": p_head_baseline,
        "priming_ratio": priming_ratio,
        "rank_modifier": rank_modifier,
        "rank_baseline": rank_baseline,
        "top 5": [(top_probs_modifier[i].item(), top_tokens_modifier[i]) for i in range(5)]
    }

    pmi_vals.append(pmi)
    prob_vals.append(p_head_given_modifier)
    pr_vals.append(priming_ratio)
    labels.append(f"{comp1} {comp2}")
    ranks_modifier.append(rank_modifier)
    
    print(f"'{comp1}' → P('{comp2}') = {p_head_given_modifier:.4f} "
          f"(PMI: {pmi}, "
          f"baseline: {p_head_baseline:.6f}, "
          f"ratio: {priming_ratio:.1f}x, "
          f"rank modifier: {rank_modifier})\n" 
          f"top5: {[(top_probs_modifier[i].item(), top_tokens_modifier[i]) for i in range(5)]}")

In [ ]:
within_5 = [rank for rank in ranks_modifier if rank <= 5]
within_10 = [rank for rank in ranks_modifier if rank <= 10]
within_25 = [rank for rank in ranks_modifier if rank <= 25]
within_50 = [rank for rank in ranks_modifier if rank <= 50]
within_100 = [rank for rank in ranks_modifier if rank <= 100]

fraction_within_5 = len(within_5) / len(ranks_modifier)
fraction_within_10 = len(within_10) / len(ranks_modifier)
fraction_within_25 = len(within_25) / len(ranks_modifier)
fraction_within_50 = len(within_50) / len(ranks_modifier)
fraction_within_100 = len(within_100) / len(ranks_modifier)

print(fraction_within_5)
print(fraction_within_10)
print(fraction_within_25)
print(fraction_within_50)
print(fraction_within_100)

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

# Colors
GREEN = "#2f9e44"
LIGHT_GREEN = "#8ce99a"

# Rank

x = np.array(pmi_vals)
y = np.array(ranks_modifier)

# Fit line
m, b = np.polyfit(x, y, 1)

x_line = np.linspace(x.min(), x.max(), 200)
y_line = m * x_line + b

plt.figure(figsize=(9, 6), dpi=400)

# Scatter
plt.scatter(x, y, color=GREEN, alpha=0.8, s=60, edgecolor="black", linewidth=0.3)

# Fit line
plt.plot(x_line, y_line, linestyle="--", color=LIGHT_GREEN, linewidth=2)

plt.xlabel("PMI")
plt.ylabel("Rank")
plt.title("Rank vs PMI")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# log Rank

x = np.array(pmi_vals)
y_log = np.log10(y + 1e-12)

# Fit line
m, b = np.polyfit(x, y_log, 1)

x_line = np.linspace(x.min(), x.max(), 200)
y_log_line = m * x_line + b

plt.figure(figsize=(9, 6), dpi=400)

# Scatter
plt.scatter(x, y_log, color=GREEN, alpha=0.8, s=60, edgecolor="black", linewidth=0.3)

# Fit line
plt.plot(x_line, y_log_line, linestyle="--", color=LIGHT_GREEN, linewidth=2)

plt.xlabel("PMI")
plt.ylabel("log Rank")
plt.title("log Rank vs PMI")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
idx1 = ranks_modifier.index(max(ranks_modifier)) # outlier
print(max(ranks_modifier))
labels[idx1]

In [ ]:
idx2 = prob_vals.index(max(prob_vals))
print(f"max prob: {max(prob_vals)}")
labels[idx2]

In [ ]:
idx3 = prob_vals.index(min(prob_vals))
print(f"min prob: {min(prob_vals)}")
labels[idx3]

In [ ]:
idx4 = pr_vals.index(max(pr_vals))
print(f"max pr: {max(pr_vals)}")
print(labels[idx4])
pmi_vals[idx4]

In [ ]:
idx5 = pr_vals.index(min(pr_vals))
print(f"min pr: {min(pr_vals)}")
print(labels[idx5])
pmi_vals[idx5]

"the matter physics" is not a gramatically correct phrase so this makes sense

In [ ]:
prob_vals2 = prob_vals.copy()
prob_vals2.remove(min(prob_vals))
pr_vals2 = pr_vals.copy()
pr_vals2.remove(min(pr_vals))

idx6 = prob_vals2.index(min(prob_vals2))
print(f"min prob: {min(prob_vals2)}")
print(labels[idx6])

idx7 = pr_vals2.index(min(pr_vals2))
print(f"min pr: {min(pr_vals2)}")
print(labels[idx7])
pmi_vals[idx7]

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

# Colors
GREEN = "#2f9e44"
LIGHT_GREEN = "#8ce99a"

# Prob 

x = np.array(pmi_vals)
y = np.array(prob_vals)

# Fit line
m, b = np.polyfit(x, y, 1)

x_line = np.linspace(x.min(), x.max(), 200)
y_line = m * x_line + b

plt.figure(figsize=(9, 6), dpi=400)

# Scatter
plt.scatter(x, y, color=GREEN, alpha=0.8, s=60, edgecolor="black", linewidth=0.3)

# Fit line
plt.plot(x_line, y_line, linestyle="--", color=LIGHT_GREEN, linewidth=2)

plt.xlabel("PMI")
plt.ylabel("P(comp2|'The comp1')")
plt.title("P(comp2|'The comp1') vs PMI")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# log Prob

x = np.array(pmi_vals)
y_log = np.log10(y + 1e-12)

m, b = np.polyfit(x, y_log, 1)

x_line = np.linspace(x.min(), x.max(), 200)
y_log_line = m * x_line + b

plt.figure(figsize=(9, 6), dpi=400)

# Scatter
plt.scatter(x, y_log, color=GREEN, alpha=0.8, s=60, edgecolor="black", linewidth=0.3)

# Fit line
plt.plot(x_line, y_log_line, linestyle="--", color=LIGHT_GREEN, linewidth=2)

plt.xlabel("PMI")
plt.ylabel("log P(comp2|'The comp1')")
plt.title("log P(comp2|'The comp1') vs PMI")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

# Colors
GREEN = "#2f9e44"
LIGHT_GREEN = "#8ce99a"

# Priming ratio 

x = np.array(pmi_vals)
y = np.array(pr_vals)

# Fit line
m, b = np.polyfit(x, y, 1)

# Create line values
x_line = np.linspace(x.min(), x.max(), 100)
y_line = m * x_line + b

# Plot
plt.figure(figsize=(9, 6), dpi=400)
plt.scatter(x, y, color=GREEN, alpha=0.8, s=60, edgecolor="black", linewidth=0.3)

# Best-fit line
plt.plot(x_line, y_line, linestyle="--", color=LIGHT_GREEN, linewidth=2)

plt.xlabel("PMI")
plt.ylabel("Priming ratio")
plt.title("Priming ratio vs. PMI")
plt.show()

# log Priming ratio 

x = np.array(pmi_vals)
y_log = np.log10(y + 1e-12)

# Fit line
m, b = np.polyfit(x, y_log, 1)

# Create line values
x_line = np.linspace(x.min(), x.max(), 100)
y_log_line = m * x_line + b

# Plot
plt.figure(figsize=(9, 6), dpi=400)
plt.scatter(x, y_log, color=GREEN, alpha=0.8, s=60, edgecolor="black", linewidth=0.3)

# Best-fit line
plt.plot(x_line, y_log_line, linestyle="--", color=LIGHT_GREEN, linewidth=2)

plt.xlabel("PMI")
plt.ylabel("log(Priming ratio)")
plt.title("log(Priming ratio) vs. PMI")
plt.show()

## Experiment 3b (Categorized)

Group bigrams by their modifier (comp1), keeping only modifiers that appear with 2+ different heads. Rerun priming analysis within these groups.

Checks: Within each modifier (comp1) group, does higher PMI -> higher model probability/better rank (of comp2)?        

In [ ]:
grouped_bigrams = defaultdict(list)

for bigram in bigrams:
    key = bigram["w1"]   # group by this field
    grouped_bigrams[key].append((bigram["w2"], bigram["pmi"], bigram["frequency"]))
    
grouped_bigrams

In [ ]:
multiple = {comp1: comp2_list for comp1, comp2_list in grouped_bigrams.items() if len(comp2_list) >= 2}
multiple

In [ ]:
len(multiple)

In [ ]:
exp3b_results = defaultdict(dict)

for comp1, comp2_list in multiple.items():
    prompt_with_modifier = f"The {comp1}"
    prompt_baseline = "The"

    with torch.no_grad():
        # Get logits for modifier
        tokens_modifier = tokenizer.encode(prompt_with_modifier)
        logits_modifier = model(torch.tensor([tokens_modifier]).to(DEVICE))
        probs_modifier = torch.softmax(logits_modifier[0, -1], dim=-1)

        # Get logits for baseline
        tokens_baseline = tokenizer.encode(prompt_baseline)
        logits_baseline = model(torch.tensor([tokens_baseline]).to(DEVICE))
        probs_baseline = torch.softmax(logits_baseline[0, -1], dim=-1)

    candidate_rows = []

    for item in comp2_list:
        comp2, pmi, frequency = item

        head_ids = tokenizer.encode(" " + comp2, add_special_tokens=False)
        if len(head_ids) != 1:
            continue

        head_token_id = head_ids[0]

        p_head_given_modifier = probs_modifier[head_token_id].item()
        p_head_baseline = probs_baseline[head_token_id].item()

        sorted_probs_modifier, sorted_indices_modifier = torch.sort(probs_modifier, descending=True)
        rank_modifier = (sorted_indices_modifier == head_token_id).nonzero().item() + 1

        sorted_probs_baseline, sorted_indices_baseline = torch.sort(probs_baseline, descending=True)
        rank_baseline = (sorted_indices_baseline == head_token_id).nonzero().item() + 1

        priming_ratio = (
            p_head_given_modifier / p_head_baseline
            if p_head_baseline > 0 else float("inf")
        )

        candidate_rows.append({
            "comp2": comp2,
            "pmi": pmi,
            "frequency": frequency,
            "head_token_id": head_token_id,
            "p_head_given_modifier": p_head_given_modifier,
            "p_head_baseline": p_head_baseline,
            "priming_ratio": priming_ratio,
            "rank_modifier": rank_modifier,
            "rank_baseline": rank_baseline,
        })

    if len(candidate_rows) < 2:
        continue

    # Rank within modifier group by PMI
    sorted_by_pmi = sorted(candidate_rows, key=lambda x: x["pmi"], reverse=True)
    for rank, row in enumerate(sorted_by_pmi, start=1):
        row["pmi_rank_within_modifier"] = rank

    # Rank within modifier group by model probability
    sorted_by_model = sorted(candidate_rows, key=lambda x: x["p_head_given_modifier"], reverse=True)
    for rank, row in enumerate(sorted_by_model, start=1):
        row["model_rank_within_candidates"] = rank

    top_pmi_head = sorted_by_pmi[0]["comp2"]
    top_model_head = sorted_by_model[0]["comp2"]

    # Store final results
    for row in candidate_rows:
        comp2 = row["comp2"]

        exp3b_results[comp1][(comp1, comp2)] = {
            "pmi": row["pmi"],
            "frequency": row["frequency"],
            "pmi_rank_within_modifier": row["pmi_rank_within_modifier"],
            "p_head_given_modifier": row["p_head_given_modifier"],
            "p_head_baseline": row["p_head_baseline"],
            "priming_ratio": row["priming_ratio"],
            "rank_modifier": row["rank_modifier"],
            "rank_baseline": row["rank_baseline"],
            "model_rank_within_candidates": row["model_rank_within_candidates"],
            "top_pmi_head": top_pmi_head,
            "top_model_head": top_model_head,
            "top1_match_with_pmi": (top_pmi_head == top_model_head),
            "num_candidates_for_modifier": len(candidate_rows),
        }

In [ ]:
exp3_results

In [ ]:
len(exp3_results)

### Bad Groups

In [ ]:
# groups with a pair such that PMI_1 > PMI_2 but rank_1 > rank_2 (theoretically, rank_1 should be < rank_2)
bad_groups = {modifier: pairs 
              for modifier, pairs in exp3_results.items() 
              if any(info1["pmi"] > info2["pmi"] and info1["rank_modifier"] > info2["rank_modifier"]
                     for pair1, info1 in pairs.items()
                     for pair2, info2 in pairs.items()
                     if pair1 != pair2)
             }

print(len(bad_groups))
bad_groups

### Good Groups

In [ ]:
good_groups = {modifier: pairs for modifier, pairs in exp3_results.items() if modifier not in bad_groups}
print(len(good_groups))
good_groups

### Statistical Tests

In [ ]:
from scipy.stats import mannwhitneyu                                                                                                                                     
                                                                                                                                                                           
def pmi_gap(pairs):
    pmis = sorted([info["pmi"] for info in pairs.values()], reverse=True)                                                                                                
    return pmis[0] - pmis[1]                                             

good_gaps = [pmi_gap(pairs) for pairs in good_groups.values()]                                                                                                           
bad_gaps = [pmi_gap(pairs) for pairs in bad_groups.values()]  

stat, p = mannwhitneyu(good_gaps, bad_gaps, alternative="greater")                                                                                                       
print(f"Good group mean PMI gap: {np.mean(good_gaps):.3f}")       
print(f"Bad group mean PMI gap: {np.mean(bad_gaps):.3f}")                                                                                                                
print(f"Mann-Whitney U={stat:.1f}, p={p:.4f}")

In [ ]:
from scipy.stats import spearmanr                                                                                                                                        
                                                                                                                                                                           
pmi_ranks = []                                                                                                                                                           
model_ranks = []                                                                                                                                                         

for modifier, pairs in exp3_results.items():
    for info in pairs.values():
        pmi_ranks.append(info["pmi_rank_within_modifier"])
        model_ranks.append(info["model_rank_within_candidates"])                                                                                                         

rho, p = spearmanr(pmi_ranks, model_ranks)                                                                                                                               
print(f"Spearman ρ={rho:.3f}, p={p:.4f}")

## PMI Extremes (Opposite)

Filter all categories of bigrams to only include two pairs maximizing PMI gap

In [ ]:
grouped_bigrams

In [ ]:
thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]  # tune this
pmi_fractions = []

for i in range(len(thresholds)):
    if i == 0:
        lb = 0.0
    else:
        lb = thresholds[i-1]
    ub = thresholds[i]
    
    filtered = {}
    
    for modifier, pairs in grouped_bigrams.items():
        if len(pairs) < 2:
            continue

        # sort by PMI
        pairs_sorted = sorted(pairs, key=lambda x: x[1])  # ascending

        low = pairs_sorted[0]
        high = pairs_sorted[-1]

        gap = high[1] - low[1]

        if gap > lb and gap <= ub:
            filtered[modifier] = [low, high]
    
    pmi_correct = 0   
    for modifier, candidates in filtered.items():
        head_low_pmi, head_high_pmi = candidates[0][0], candidates[1][0]

        p_low = exp3_results[modifier][(modifier, head_low_pmi)]["p_head_given_modifier"]                                                                                    
        p_high = exp3_results[modifier][(modifier, head_high_pmi)]["p_head_given_modifier"]                                                                                  

        if p_high > p_low:
            pmi_correct += 1                                                                                                                                                                                                                                                                                           
    
    pmi_fractions.append(pmi_correct/len(filtered))
    
    print(f"higher PMI -> higher prob {lb, ub}: {pmi_correct}/{len(filtered)} ({pmi_correct/len(filtered):.3f})")

In [ ]:
# x-axis labels (bins)
labels = []
for i in range(len(thresholds)):
    if i == 0:
        lb = 0.0
    else:
        lb = thresholds[i-1]
    ub = thresholds[i]
    labels.append(f"{lb}-{ub}")

plt.figure(figsize=(10, 6), dpi=400)
plt.bar(labels, pmi_fractions)

plt.xlabel("PMI Gap Range")
plt.ylabel("Fraction model prefers higher-PMI head")
plt.title("Model Preference vs PMI Gap")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()